# B1 · Transfer learning et modèles pré-entraînés — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b1_transfer/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B1.Q1 à B1.Q11, B1.R1 à B1.R3, B1.1 à B1.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B1.11 à B1.16 | charger et sonder : trois modèles pré-entraînés, un ResNet face aux images de 32 pixels, embeddings figés, linear probe et k plus proches voisins, résolution, projections, sondage bloc par bloc (*prochaine session de génération*) | | | |
| B | B1.17 à B1.19 | fine-tuner : le dernier bloc avec des learning rates par groupe, un fine-tuning qui détruit le modèle, le fine-tuning complet (*prochaine session de génération*) | | | |
| C | B1.20 à B1.23 | quand le transfert aide ou échoue : courbes d'apprentissage, domaine lointain, oubli catastrophique (*prochaine session de génération*) | | | |
| D | B1.24 à B1.28 | modèles de fondation, coût et livraison : zéro-shot avec CLIP, few-shot par prototypes, latence sur CPU, publication du modèle dérivé, et le défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Charger un modèle pré-entraîné avec son prétraitement (torchvision, Hugging Face Hub) et lire sa fiche modèle et sa licence.
- Entraîner un linear probe et un k plus proches voisins sur des embeddings figés, et sonder les blocs d'un backbone.
- Fine-tuner un réseau : dégel progressif, learning rates par groupe, warmup, batchnorm maîtrisée.
- Mesurer le gain du pré-entraînement par des courbes d'apprentissage, et diagnostiquer transfert négatif et oubli catastrophique.
- Classer en zéro-shot et en few-shot avec CLIP et DINOv2, et comparer des backbones sur CPU.

**Rappel express.** Linear probe : $\hat{\mathbf{y}} = \operatorname{softmax}(\mathbf{W}\,\phi(\mathbf{x}) + \mathbf{b})$, $d \times K + K$ paramètres. Learning rates discriminatifs : $\eta_\ell = \eta_{\text{tête}}\,\gamma^{L-\ell}$ ; warmup : $\eta_t = \eta_{\max}\,\min(1, t / t_{\text{warmup}})$. L2-SP : $\Omega = \frac{\lambda}{2}\lVert\theta - \theta_0\rVert^2$. Zéro-shot : $p(c \mid \mathbf{x}) \propto \exp(s \cos(\mathbf{u}_x, \mathbf{v}_c))$. Prototype : $\mathbf{m}_c = \frac{1}{|S_c|}\sum_{\mathbf{x} \in S_c} \phi(\mathbf{x})$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B1.1 à B1.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; plusieurs lettres : entre guillemets (`"AC"`) ; une forme de tenseur : entre guillemets (`"(2, 3)"`) ; un nom de classe : entre guillemets (`"chat"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B1.6, l'estimation 🧮 B1.7, la réflexion (🗣️ B1.8, ⚖️ B1.9, 📄 B1.10) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np


# ✏️ B1.1: side of a map after a convolution or a pooling, floor((n + 2p - k) / s) + 1 (ch. 21)
def out_side(n, k, s, p):
    return (n + 2 * p - k) // s + 1


def resnet18_sides(n):
    """Sides after the stem, layer1, layer2, layer3 and layer4 of a ResNet-18 for an n x n image."""
    stem = out_side(out_side(n, 7, 2, 3), 3, 2, 1)          # conv1, then maxpool
    l2 = out_side(stem, 3, 2, 1)                             # layer1 keeps the size; layer2 to 4 start with stride 2
    l3 = out_side(l2, 3, 2, 1)
    return {"stem": stem, "layer1": stem, "layer2": l2, "layer3": l3, "layer4": out_side(l3, 3, 2, 1)}


def shape(*axes):
    return "(" + ", ".join(str(a) for a in axes) + ")"


S128_B11 = resnet18_sides(128)

# ✏️ B1.4: parameters of ResNet-18 (convolutions without bias; gamma and beta in each batchnorm)
def conv_weights(c_in, c_out, k):
    return c_in * c_out * k * k


def bn_params(c):
    return 2 * c


BLOCK0_B14 = (conv_weights(256, 512, 3) + bn_params(512) + conv_weights(512, 512, 3) + bn_params(512)
              + conv_weights(256, 512, 1) + bn_params(512))
BLOCK1_B14 = 2 * (conv_weights(512, 512, 3) + bn_params(512))
LAYER4_B14 = BLOCK0_B14 + BLOCK1_B14
IMAGENET_B14 = 11_689_512                                    # weights.meta["num_params"], ImageNet head included
HEAD10_B14 = 512 * 10 + 10
NET10_B14 = IMAGENET_B14 - (512 * 1000 + 1000) + HEAD10_B14
TRAIN_B14 = LAYER4_B14 + HEAD10_B14

# ✏️ B1.2: six embeddings; ties of votes go to the first class in alphabetical order (KNeighborsClassifier)
E_B12 = np.array([[3, 0, 4], [0, 3, 4], [16, 2, 8], [2, 1, 2], [1, 0, 0], [0, 1, 1]], dtype=float)
Y_B12 = np.array(["oiseau", "avion", "avion", "oiseau", "avion", "oiseau"])
Q_B12 = np.array([1, 2, 2], dtype=float)
COS_B12 = E_B12 @ Q_B12 / (np.linalg.norm(E_B12, axis=1) * np.linalg.norm(Q_B12))


def knn_cosine_b12(k):
    nearest = np.argsort(-COS_B12, kind="stable")[:k]
    classes, votes = np.unique(Y_B12[nearest], return_counts=True)      # classes in alphabetical order
    return str(classes[np.argmax(votes)])                                # argmax: the first of the tied classes


# ✏️ B1.3: discriminative learning rates (head, layer4, layer3, layer2), warmup then cosine, in units of 1e-5
HEAD_LR_B13, GAMMA_B13, T_WARMUP_B13, T_TOTAL_B13 = 100.0, 0.3, 5, 105


def factor_b13(t):
    if t < T_WARMUP_B13:
        return t / T_WARMUP_B13
    progress = (t - T_WARMUP_B13) / (T_TOTAL_B13 - T_WARMUP_B13)
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))


def lr_b13(depth, t):
    """depth 0 = the head, 1 = layer4, 2 = layer3, 3 = layer2 (that is, L - l)."""
    return HEAD_LR_B13 * GAMMA_B13 ** depth * factor_b13(t)


# ✏️ B1.5: zero-shot CLIP, classes (A) airplane, (B) bird, (C) ship, (D) truck
COS_B15 = np.array([0.24, 0.30, 0.28, 0.19])


def zero_shot_b15(s, cos=COS_B15):
    z = s * cos
    e = np.exp(z - z.max())
    return e / e.sum()


P100_B15 = zero_shot_b15(math.exp(4.60517))
PINIT_B15 = zero_shot_b15(math.exp(2.6592))

# The same answers from the libraries (no download: weights=None gives the architecture)
import torch
from sklearn.neighbors import KNeighborsClassifier
from torch import nn
from torchvision.models import resnet18

net = resnet18(weights=None).eval()
assert sum(p.numel() for p in net.parameters()) == IMAGENET_B14
net.fc = nn.Linear(net.fc.in_features, 10)
assert sum(p.numel() for p in net.parameters()) == NET10_B14
assert sum(p.numel() for p in net.layer4.parameters()) == LAYER4_B14
for size in (128, 32, 100):
    with torch.inference_mode():
        h = net.maxpool(net.relu(net.bn1(net.conv1(torch.zeros(16, 3, size, size)))))
        sides = {"stem": h.shape[-1]}
        for name in ("layer1", "layer2", "layer3", "layer4"):
            h = getattr(net, name)(h)
            sides[name] = h.shape[-1]
        logits = net(torch.zeros(16, 3, size, size))
    assert sides == resnet18_sides(size) and tuple(logits.shape) == (16, 10), (size, sides)

for k in range(1, 7):
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine").fit(E_B12, Y_B12)
    assert knn.predict(Q_B12[None])[0] == knn_cosine_b12(k), k

groups = [nn.Parameter(torch.zeros(1)) for _ in range(4)]
optimizer = torch.optim.AdamW([{"params": [p], "lr": 1e-3 * GAMMA_B13 ** d} for d, p in enumerate(groups)])
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, factor_b13)
for t in range(60):
    assert np.allclose([pg["lr"] / 1e-5 for pg in optimizer.param_groups], [lr_b13(d, t) for d in range(4)])
    optimizer.step()
    scheduler.step()

assert np.allclose(torch.softmax(torch.exp(torch.tensor(4.60517)) * torch.tensor(COS_B15), 0).numpy(), P100_B15)
print("✅ the paper answers agree with PyTorch, torchvision and scikit-learn")

✅ the paper answers agree with PyTorch, torchvision and scikit-learn


**B1.Q1 — Réutiliser plutôt que repartir de zéro : trois raisons et une limite**

In [3]:
wb.record("B1.Q1a", "ABD", mistakes={"un modèle pré-entraîné hérite des biais de ses données d'origine (fiche §201.7)": "ABCD",
          "une raison juste manque : relis ce que savent déjà faire les premières couches": "AB",
          "une raison juste manque : relis ce que coûte l'entraînement d'un grand réseau": "AD",
          "une raison juste manque : relis combien d'images étiquetées demande un ResNet entraîné de zéro": "BD"})
wb.record("B1.Q1b", 400, mistakes={"on demande un nombre de millions : 1 million s'écrit 1": 400_000_000,
          "142 millions, c'est le dataset de DINOv2": 142,
          "1,28 million, ce sont les images d'ImageNet-1k": 1},
          fractional="on demande un nombre entier de millions de paires")
wb.record("B1.Q1c", False, mistakes={"relis le §201.1 : ce que 500 images font à un réseau de millions de paramètres (ch. 9)": True})
wb.record("B1.Q1d", "B", mistakes={"un modèle pré-entraîné tourne sur un CPU : tout le notebook du chapitre le fait": "A", "remplacer la tête est justement le premier geste du transfert": "C"})
wb.record("B1.Q1e", True, mistakes={"relis le §201.1 : qui a payé les jours de calcul sur des GPU ?": False})

WB_ANSWER {"hash": "16640f2bcaf1629c674683e41d06f60775a5a79d66722d703e612a43f2b35127", "id": "B1.Q1a", "kind": "str", "mistakes": {"22420921b01d4027617b1bc3811c82dc0c9d5b65c879f8e736b47d4d6fbbbd9a": "une raison juste manque : relis ce que savent déjà faire les premières couches", "5ef8dd70c0dcb708f00835bd895885d95cf3daa61c67a7acf43b7583f0418b08": "une raison juste manque : relis combien d'images étiquetées demande un ResNet entraîné de zéro", "cc03147268db07b7a759be24f111bcff03bcb2ad921ffffa63ba585443708a16": "une raison juste manque : relis ce que coûte l'entraînement d'un grand réseau", "cf65a337a0fffea0eb64a5a2dabcc33e1512fb699d8fbf013374ce85183bdad1": "un modèle pré-entraîné hérite des biais de ses données d'origine (fiche §201.7)"}}
📝 Ex B1.Q1a : réponse enregistrée (str).
WB_ANSWER {"fractional": "on demande un nombre entier de millions de paires", "hash": "aefc7f50df753c0139324cfcac73ba7de40d0ad96726bea9a9cc874f334217b1", "id": "B1.Q1b", "kind": "int", "magnitude": 2, "mistakes"

**B1.Q2 — Premières couches génériques, dernières couches spécialisées : vrai ou faux**

In [4]:
wb.record("B1.Q2a", True, mistakes={"relis la hiérarchie des features (§201.1) : on retrouve ces filtres dans presque tous les CNN": False})
wb.record("B1.Q2b", False, mistakes={"relis la hiérarchie des features : où se trouvent les features les plus liées aux 1 000 classes ?": True})
wb.record("B1.Q2c", True, mistakes={"relis la règle qui suit la hiérarchie des features (§201.1)": False})
wb.record("B1.Q2d", False, mistakes={"relis le §201.7 : quelles couches deviennent les premières inutiles sur un domaine lointain ?": True})
wb.record("B1.Q2e", True, mistakes={"relis d'où vient le mot « probe » (§201.3)": False})

WB_ANSWER {"hash": "bad9b271f1a8807a5a9d7e0a8c913d59afa55632f8168210fa0ec9eef9ba44ea", "id": "B1.Q2a", "kind": "bool", "mistakes": {"5002d72c828adefb2a30594b56c604d996ab91ded8a38ab47c8f949aff4663dd": "relis la hiérarchie des features (§201.1) : on retrouve ces filtres dans presque tous les CNN"}}
📝 Ex B1.Q2a : réponse enregistrée (bool).
WB_ANSWER {"hash": "1ed9c4d950a112778f3c48ec31864b7797e940e9b05558b2dc6066a81ec5bc19", "id": "B1.Q2b", "kind": "bool", "mistakes": {"56d1bc3604f40febf2171fdb049ea83b20a857bb7702af28185f6c69fd7c8713": "relis la hiérarchie des features : où se trouvent les features les plus liées aux 1 000 classes ?"}}
📝 Ex B1.Q2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "b88336e0d79b28b997f90a5c5c98079a4995c06b16f758ba98b7d36a3c5c18fb", "id": "B1.Q2c", "kind": "bool", "mistakes": {"7bddb03bc86850370ade8b69e3e022bd206b6b8f86dda2917c9f5c86d075c07e": "relis la règle qui suit la hiérarchie des features (§201.1)"}}
📝 Ex B1.Q2c : réponse enregistrée (bool).
WB_ANSWER 

**B1.Q3 — Où trouver un modèle pré-entraîné, et quoi lire avant de s'en servir**

In [5]:
wb.record("B1.Q3a", "C", mistakes={"torchvision n'a pas DINOv2 : il vient du Hugging Face Hub": "A", "torch.load relit un fichier déjà sur le disque, il ne télécharge rien": "B"})
wb.record("B1.Q3b", "ABC", mistakes={"la fiche modèle décrit le modèle, pas ta machine ni ta connexion": "ABCD",
          "une rubrique de la fiche modèle manque : ce qu'on a le droit d'en faire": "BC",
          "une rubrique de la fiche modèle manque : d'où viennent ses données": "AC",
          "une rubrique de la fiche modèle manque : les usages hors du champ prévu (out of scope)": "AB"})
wb.record("B1.Q3c", False, mistakes={"relis l'encadré ⚠️ « public n'est pas libre » (§201.2)": True})
wb.record("B1.Q3d", "B", mistakes={"c'était la licence de la sortie, en avril 2023 : relis le tableau du §201.2": "A", "c'est CLIP qui n'affiche aucune licence, pas DINOv2": "C"})
wb.record("B1.Q3e", True, mistakes={"relis l'encadré sur l'API transformers : revision= désigne un commit du dépôt": False})

WB_ANSWER {"hash": "8479c131585e69a5581397157f3dbd2bb5e9c1ef3569c1153a564899bc257c6b", "id": "B1.Q3a", "kind": "str", "mistakes": {"7e29fdeed51acc19e73114fdaf751f683945085941f7877002e6af2b902f47c5": "torch.load relit un fichier déjà sur le disque, il ne télécharge rien", "8a434913d45e4d6e0b1b1dd279c79f588ed237db36ac8fc9efa8fe51e378353a": "torchvision n'a pas DINOv2 : il vient du Hugging Face Hub"}}
📝 Ex B1.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "c03347ac69d30b23e523cb591785bddc274b0fd2cb74e10eeb12e0258711317c", "id": "B1.Q3b", "kind": "str", "mistakes": {"0c62f588ecc4ea074a350b5a6d9adca65af2bae331b10a66032c2e8192acf146": "la fiche modèle décrit le modèle, pas ta machine ni ta connexion", "1995e106d89e33e48c79838c9d2a6992df792d136d7370d59e2a797d1aec04fa": "une rubrique de la fiche modèle manque : ce qu'on a le droit d'en faire", "67dabb9accc71b264bf81e2b8024a0fb202305ca155006b558b7c47097aa5822": "une rubrique de la fiche modèle manque : les usages hors du champ prévu (out o

**B1.Q4 — weights.transforms() : pourquoi le prétraitement fait partie du modèle**

In [6]:
wb.record("B1.Q4a", "C", mistakes={"la normalisation se fait sur des valeurs déjà dans [0, 1] : elle ne peut pas venir en premier": "A", "on ne recadre pas avant d'avoir redimensionné : relis les quatre étapes de l'encadré": "B"})
wb.record("B1.Q4b", False, mistakes={"les formes ne changent pas quand on oublie la normalisation : relis le §201.5": True})
wb.record("B1.Q4c", 232, mistakes={"224 est la taille du recadrage (crop_size), pas du redimensionnement": 224,
          "256 est le redimensionnement d'autres jeux de poids : relis l'encadré sur weights.transforms()": 256})
wb.record("B1.Q4d", "A", mistakes={"chaque modèle a ses statistiques : CLIP n'a pas été entraîné avec celles d'ImageNet": "B", "on normalise avec les statistiques du pré-entraînement, pas celles de tes données": "C"})
wb.record("B1.Q4e", True, mistakes={"compare les deux jeux de poids de ResNet-50 cités dans l'encadré du §201.2": False})

WB_ANSWER {"hash": "db3e5be079f0e000ce037fa84a599e7058a464ead30bed4dec022a4307f33f0f", "id": "B1.Q4a", "kind": "str", "mistakes": {"0a23bb9952296c11052bfff00ef8e32c4d3d8ec3ac953c98dd44d7ee17eef88f": "la normalisation se fait sur des valeurs déjà dans [0, 1] : elle ne peut pas venir en premier", "796cc465541b67e53548cefed782cb1d6ec446837283f6ff3ef37e12c7561e4f": "on ne recadre pas avant d'avoir redimensionné : relis les quatre étapes de l'encadré"}}
📝 Ex B1.Q4a : réponse enregistrée (str).
WB_ANSWER {"hash": "f76d6a9b6e099a3562da2e5d158476227ff332c1a0d40b0a83ecff13037e1a89", "id": "B1.Q4b", "kind": "bool", "mistakes": {"0202a5856e6a35ae3ae3581ba1617e18035631ed3600156dd4a0f42c762bf724": "les formes ne changent pas quand on oublie la normalisation : relis le §201.5"}}
📝 Ex B1.Q4b : réponse enregistrée (bool).
WB_ANSWER {"hash": "b1bc8161de56f11e3008b633dfe9eba5567bb842b3e68deb5bb7044ce5240740", "id": "B1.Q4c", "kind": "int", "magnitude": 2, "mistakes": {"df43b51f6f40ea24e7dd59f3df7e3dfb87

**B1.Q5 — Extraction de features, linear probe, fine-tuning : qui s'entraîne ?**

In [7]:
wb.record("B1.Q5a", "A", mistakes={"le backbone reste gelé en extraction de features": "B", "le backbone, batchnorms comprises, ne change pas en extraction de features": "C"})
wb.record("B1.Q5b", "C", mistakes={"tout le réseau, c'est le fine-tuning complet": "A", "la tête seule, backbone gelé, c'est l'extraction de features": "B"})
wb.record("B1.Q5c", True, mistakes={"un k-NN garde les exemples et compare : y a-t-il des poids à régler ?": False})
wb.record("B1.Q5d", 768 * 7 + 7, mistakes={"n'oublie pas les biais, un par classe": 768 * 7,
          "chaque classe a un poids par composante de l'embedding": 768 + 7})
wb.record("B1.Q5e", False, mistakes={"un embedding figé ne change jamais : relis le §201.3": True})

WB_ANSWER {"hash": "f59b00525e1ccb832aca87153154966366bc3aac5b5ceeb6d09a04dc7a52dd29", "id": "B1.Q5a", "kind": "str", "mistakes": {"1570283aca0887dfa9795f98e75785d315b56756fc64c94e282c93380f83d3e4": "le backbone, batchnorms comprises, ne change pas en extraction de features", "f338b7d3f215b97376658b26d172f71031cba66ba01813edcb1abfd813bd176b": "le backbone reste gelé en extraction de features"}}
📝 Ex B1.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "b214a0fcf44bda394b87b8d98e5e098c0d1cc86dc6bee9ea57a2433b502fb790", "id": "B1.Q5b", "kind": "str", "mistakes": {"0d5a00c3ff8386af7d504b547c1fec9f9e968ec7f73050a6abb873b1e36f23ea": "tout le réseau, c'est le fine-tuning complet", "4143092b98e8333771ac788bd1a9fbcc872ace27cb676514b46f294940856e2d": "la tête seule, backbone gelé, c'est l'extraction de features"}}
📝 Ex B1.Q5b : réponse enregistrée (str).
WB_ANSWER {"hash": "ebec271a9ec131800b175b3f025793c0c3e1e8992bcc755529cc21246184e7a5", "id": "B1.Q5c", "kind": "bool", "mistakes": {"8af6153

**B1.Q6 — Pourquoi un learning rate 10 à 100 fois plus petit pour fine-tuner ?**

In [8]:
wb.record("B1.Q6a", "A", mistakes={"AdamW n'a pas de limite de ce genre : la raison tient aux poids de départ": "B", "un plus petit learning rate fait des pas plus courts : il ne converge pas plus vite": "C"})
wb.record("B1.Q6b", True, mistakes={"relis le §201.4 : que vaut la loss quand la tête est encore aléatoire ?": False})
wb.record("B1.Q6c", "B", mistakes={"LP-FT commence par la tête : relis l'ordre des deux lettres": "A", "LP-FT fait les deux étapes l'une après l'autre, une seule fois": "C"})
wb.record("B1.Q6d", False, mistakes={"relis la formule des learning rates discriminatifs : vers quelle extrémité décroissent-ils ?": True})
wb.record("B1.Q6e", "C", mistakes={"le warmup change le learning rate, pas la mémoire": "A", "le warmup ne touche pas aux batchnorms": "B"})

WB_ANSWER {"hash": "8867469765df856ae8f684fe059ed69cf0f7ff59d5571f9afd23afe24afe47d1", "id": "B1.Q6a", "kind": "str", "mistakes": {"029f1c2800511e81058bf53a4f6e76247d47e01ec8594db180a57a0d803ba799": "un plus petit learning rate fait des pas plus courts : il ne converge pas plus vite", "a49336600e1a461fc3a6a8d3bcbb3bd866bcde75b62dc583f71ff39ef22174cc": "AdamW n'a pas de limite de ce genre : la raison tient aux poids de départ"}}
📝 Ex B1.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "10376e60d51f54785d84cd0f0d381c8ef201226a05553005b410cd7b9b9a79ca", "id": "B1.Q6b", "kind": "bool", "mistakes": {"9f7527b5241cfb58b2fa3111d407ad0df00f8e761d7974661cdb5c398b1d1148": "relis le §201.4 : que vaut la loss quand la tête est encore aléatoire ?"}}
📝 Ex B1.Q6b : réponse enregistrée (bool).
WB_ANSWER {"hash": "2c69d7d4b51a96765ba42a2d324f18e40d84189f86533cf8d68f2d83199fecf0", "id": "B1.Q6c", "kind": "str", "mistakes": {"5c9e1eba33fed5d83b724aefd2a984e8b7bbbcba60e13f2fcfe0d867d3c1e76a": "LP-FT fai

**B1.Q7 — Batchnorm pendant le fine-tuning sur de tout petits batches**

In [9]:
wb.record("B1.Q7a", "B", mistakes={"les statistiques glissantes servent en mode eval(), pas en mode train()": "A", "une batchnorm calcule ses statistiques, elle ne les fixe pas": "C"})
wb.record("B1.Q7b", True, mistakes={"relis les deux régimes de la batchnorm (§201.4)": False})
wb.record("B1.Q7c", False, mistakes={"relis la remarque sur l'ordre des appels sous freeze_batchnorm (§201.4)": True})
wb.record("B1.Q7d", "C", mistakes={"les formes ne dépendent pas du mode de la batchnorm": "A", "des statistiques bruitées n'empêchent pas le calcul : le problème apparaît à l'évaluation": "B"})
wb.record("B1.Q7e", True, mistakes={"eval() change la façon de normaliser, pas requires_grad : relis la parade du §201.4": False})

WB_ANSWER {"hash": "9173f8ee21a0ffa0a967f144fd9448e39c5fe3d6ce169dcb26782aeda909de22", "id": "B1.Q7a", "kind": "str", "mistakes": {"b9f02e027e790d899b3e093177eb38562a8d1d72868b5dcac5921c2aead56f76": "une batchnorm calcule ses statistiques, elle ne les fixe pas", "cf06ce7ddeb33a970d8c9c87bcc2d0235391b0561dbeb50020fa305422a3b423": "les statistiques glissantes servent en mode eval(), pas en mode train()"}}
📝 Ex B1.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "062925896948499051288232238da0627d96067fb4a2abc7f463e339eba18b54", "id": "B1.Q7b", "kind": "bool", "mistakes": {"d8f7e3a6c9f81c602bd9752c4840d3c32052062886890c8f39e0d731817a64d7": "relis les deux régimes de la batchnorm (§201.4)"}}
📝 Ex B1.Q7b : réponse enregistrée (bool).
WB_ANSWER {"hash": "e954eb5032f90f7c1f25d9c4bfe3a2480d8fa658690e49403e68ee3661a9fc7d", "id": "B1.Q7c", "kind": "bool", "mistakes": {"e6fce9bfd2f161104efbd11744053faca434b480c94947579324c91a47889e02": "relis la remarque sur l'ordre des appels sous freeze_batc

**B1.Q8 — Peu ou beaucoup de données, domaine proche ou lointain : la bonne stratégie**

In [10]:
wb.record("B1.Q8a", "A", mistakes={"le learning rate d'un entraînement de zéro détruit les features pré-entraînées": "B", "300 images ne suffisent pas à entraîner un ResNet de zéro": "C"})
wb.record("B1.Q8b", "B", mistakes={"des pixels bruts ne forment pas un bon embedding": "A", "avec 200 000 images d'un domaine proche, fine-tuner rapporte davantage": "C"})
wb.record("B1.Q8c", "C", mistakes={"des spectrogrammes sont loin des photos d'ImageNet : ce n'est pas le cas facile": "A", "150 images ne suffisent pas à entraîner un ResNet-50 de zéro": "B"})
wb.record("B1.Q8d", True, mistakes={"relis la case « beaucoup de données, domaine lointain » de la matrice de décision": False})
wb.record("B1.Q8e", False, mistakes={"relis le protocole des courbes d'apprentissage (§201.6) : que vaut une mesure sur 10 images tirées une fois ?": True})

WB_ANSWER {"hash": "289f29c8a0757345dc5fbf6d481ba866cb868fb6c01e16dac11aaf402e94cfce", "id": "B1.Q8a", "kind": "str", "mistakes": {"7c778f0ad9328b583f8359e9eb5d09e03b2ce882b7e9c66b61d454ed18434972": "300 images ne suffisent pas à entraîner un ResNet de zéro", "8ba019e5dd89fa88cd91c23b4461149914d6ede11e9e7dc893638f78c483195e": "le learning rate d'un entraînement de zéro détruit les features pré-entraînées"}}
📝 Ex B1.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "9840144cf5cb7b72c6c9e2495d8c30c07bad2772cc649d98550d281a399b20ce", "id": "B1.Q8b", "kind": "str", "mistakes": {"8dadd170273574a369337771ef61f2449591a509f933dbe97c7ca967f6295095": "avec 200 000 images d'un domaine proche, fine-tuner rapporte davantage", "91bd1c565ea21ba4ca53f0a3a8fcd3027c3ac7dc488bb09a7d8ab46031c68710": "des pixels bruts ne forment pas un bon embedding"}}
📝 Ex B1.Q8b : réponse enregistrée (str).
WB_ANSWER {"hash": "2c5079ef2b4ae852ddb4a124d64c8bd244a56c73956ffed6020b1dd1def3872a", "id": "B1.Q8c", "kind": "s

**B1.Q9 — Transfert négatif et oubli catastrophique : reconnaître les symptômes**

In [11]:
wb.record("B1.Q9a", "B", mistakes={"une accuracy qui baisse d'une epoch à l'autre, c'est de l'overfitting ou du bruit": "A", "le transfert négatif compare deux résultats finaux, pas des gradients": "C"})
wb.record("B1.Q9b", "A", mistakes={"le modèle va bien sur sa nouvelle tâche : c'est l'ancienne qui est perdue": "B", "l'overfitting compare l'entraînement et le test d'une même tâche": "C"})
wb.record("B1.Q9c", True, mistakes={"relis le §201.7 : à quoi compare-t-on un transfert pour dire qu'il nuit ?": False})
wb.record("B1.Q9d", False, mistakes={"relis les parades contre l'oubli (§201.7)": True})
wb.record("B1.Q9e", "C", mistakes={"L2-SP tire les poids vers leurs valeurs pré-entraînées": "A", "des learning rates discriminatifs bougent peu les premières couches": "B"})

WB_ANSWER {"hash": "9217fac393414dffeeb2f92f3235385cd84e87be0159f05c410385686bc0031e", "id": "B1.Q9a", "kind": "str", "mistakes": {"1790f354883a170690d619c28a3eab723bb76496d392bd61e343e9e629dee406": "une accuracy qui baisse d'une epoch à l'autre, c'est de l'overfitting ou du bruit", "aa8b217bbbe80a547cafddc30eee8689ab146fc29dc214aba164cda8c14b988d": "le transfert négatif compare deux résultats finaux, pas des gradients"}}
📝 Ex B1.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "69daf62a2788386a70a1236a94ec3a98e7a18bf62f571308c7c90e2b169e37bc", "id": "B1.Q9b", "kind": "str", "mistakes": {"044566b014e8672095d5103752200bd6fe13292179dc0e7ed10c9eedc8e6434a": "le modèle va bien sur sa nouvelle tâche : c'est l'ancienne qui est perdue", "8905f7ba7fb8b2411d01a90cd2385f99e1c9292641c2ab1dbe7633e4f28237d0": "l'overfitting compare l'entraînement et le test d'une même tâche"}}
📝 Ex B1.Q9b : réponse enregistrée (str).
WB_ANSWER {"hash": "151d99e58b7c1a1b13f3492a3876c0b164ecd96b4f1eb76f162ff6b6ef0

**B1.Q10 — Classer sans exemple d'entraînement : le zéro-shot de CLIP**

In [12]:
wb.record("B1.Q10a", "A", mistakes={"le zéro-shot n'utilise aucune image étiquetée": "B", "CLIP n'a pas de tête ImageNet : il fabrique sa tête à partir de mots": "C"})
wb.record("B1.Q10b", False, mistakes={"relis le §201.8 : la valeur de config.json est celle du départ de l'entraînement": True})
wb.record("B1.Q10c", True, mistakes={"calcule l'écart entre deux logits quand s = 1 : que fait une softmax de nombres si proches ?": False})
wb.record("B1.Q10d", "C", mistakes={"un ensemble de prompts porte sur les phrases des classes, pas sur les images": "A", "un seul CLIP suffit : on moyenne des embeddings de texte": "B"})
wb.record("B1.Q10e", False, mistakes={"relis le §201.8 : sur quoi CLIP a-t-il été entraîné ?": True})

WB_ANSWER {"hash": "024b0f0cb4c82fc8c3b219a5b10c1d8c8b5a7c9490f4f1cfb97f161e8b29b4ae", "id": "B1.Q10a", "kind": "str", "mistakes": {"2d2f2bcb31d675356934da3068de923a4fa9c79aa725b7f7a75086c677d9e426": "le zéro-shot n'utilise aucune image étiquetée", "d7a8919887ee41931b6c7c9d69884a745c310c2fa3d96bb3dc7b506aa5be7b1b": "CLIP n'a pas de tête ImageNet : il fabrique sa tête à partir de mots"}}
📝 Ex B1.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "96505bdb33428d90d26e3a3a31d847388ac98125bd9688a96e4d356f7c445835", "id": "B1.Q10b", "kind": "bool", "mistakes": {"00d764021c46d58b7ec967a598ac9bc90418cc6d7d536c387c5f7e14e8982dbf": "relis le §201.8 : la valeur de config.json est celle du départ de l'entraînement"}}
📝 Ex B1.Q10b : réponse enregistrée (bool).
WB_ANSWER {"hash": "6641c3920e6e6b9852675ada0f54be4a56299b7822d2f52b219c09060388c117", "id": "B1.Q10c", "kind": "bool", "mistakes": {"e69501922a9348cf86742df7ebe46d0ae18c0545a5df26f82edc2b3c05a115fb": "calcule l'écart entre deux logits qua

**B1.Q11 — Auto-supervisé (DINOv2) ou supervisé sur ImageNet : quels embeddings ?**

In [13]:
wb.record("B1.Q11a", "B", mistakes={"rapprocher une image de sa légende, c'est l'apprentissage contrastif de CLIP": "A", "DINOv2 n'utilise aucun label, ni ceux d'ImageNet ni d'autres": "C"})
wb.record("B1.Q11b", True, mistakes={"relis le §201.8 : d'où viennent les modèles small, base et large ?": False})
wb.record("B1.Q11c", "A", mistakes={"pixel_values est l'entrée du modèle, pas sa sortie": "B", "ces tokens donnent un vecteur par morceau d'image, pas un par image": "C"})
wb.record("B1.Q11d", False, mistakes={"relis la fin du §201.9 : le classement ImageNet est un indice, pas une preuve": True})
wb.record("B1.Q11e", (448 // 14) ** 2 + 1, mistakes={"n'oublie pas le token CLS": (448 // 14) ** 2,
          "257 tokens, c'est pour 224 × 224 : relis l'énoncé, le tenseur est donné tel quel au modèle": 257,
          "compte les carrés sur toute l'image, pas sur un seul côté": 448 // 14 + 1})

WB_ANSWER {"hash": "3cfeaaea53e50e0057b9f341d69b83ca861f4f59b7c5dbd18743ef32d1a61029", "id": "B1.Q11a", "kind": "str", "mistakes": {"2a6c16243d1d78d91b9e6e69f983d5efafdac72b51adc187401a86a917aa6ae1": "DINOv2 n'utilise aucun label, ni ceux d'ImageNet ni d'autres", "714d3631f35dc4f86ffeaccbbf63b8b3ba2ed505133f039379851e08bbce415e": "rapprocher une image de sa légende, c'est l'apprentissage contrastif de CLIP"}}
📝 Ex B1.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "ef0341a78db7c85bd4a511147194e6204411880cd05d3c3919afeac1015a4876", "id": "B1.Q11b", "kind": "bool", "mistakes": {"3d8fe478bd06aa534a510aa133d3b698263044c4489c9189f6be3bf636495596": "relis le §201.8 : d'où viennent les modèles small, base et large ?"}}
📝 Ex B1.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "694c6b6824229ba4bb532b729d505f8766c3cb59a01fdf3b2d36ae25485b3afa", "id": "B1.Q11c", "kind": "str", "mistakes": {"a388f6c5d2a160d24e3b01b349e38c5d88ccb7fef4cff0d65059662d5fc52b17": "pixel_values est l'entrée du 

**B1.R1 — Ch. 29 : licence d'un dataset et décalage de distribution**

In [14]:
wb.record("B1.R1a", False, mistakes={"NC, c'est « non commercial » (ch. 29)": True})
wb.record("B1.R1b", "A", mistakes={"ImageNet ne vend pas de licence commerciale : relis l'encadré 🕰️ sur les licences des poids": "B", "certaines images d'ImageNet sont protégées par le droit d'auteur": "C"})
wb.record("B1.R1c", "C", mistakes={"un découpage non stratifié change les proportions des classes, pas la distribution d'usage": "A", "un planning de learning rate n'a rien à voir avec les données": "B"})
wb.record("B1.R1d", True, mistakes={"relis le §201.7, deuxième paragraphe": False})
wb.record("B1.R1e", "B", mistakes={"plus d'epochs ne disent rien des données d'usage": "A", "un découpage du même dataset suit la même distribution : il ne voit pas le décalage": "C"})

WB_ANSWER {"hash": "52116e509350caa39b6cd68b9d07047f2176f1d855fe9cc6c1590113f498672f", "id": "B1.R1a", "kind": "bool", "mistakes": {"3ea71dd2f2fec0764f62c78745c162e53efc8825ccf1f8719eaeb49ef90131f8": "NC, c'est « non commercial » (ch. 29)"}}
📝 Ex B1.R1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "c7ac0392fae70043cab132a531156343dcb9945b3a79d5d289bba79424a88dd1", "id": "B1.R1b", "kind": "str", "mistakes": {"c3617d32180ba37c974bdadfabfee52c664940bde624a41732899831b57995e2": "ImageNet ne vend pas de licence commerciale : relis l'encadré 🕰️ sur les licences des poids", "fd28dd813f2cc766e45a68997d7a7326d7ac7229242d73382971161313779773": "certaines images d'ImageNet sont protégées par le droit d'auteur"}}
📝 Ex B1.R1b : réponse enregistrée (str).
WB_ANSWER {"hash": "eedcb8e6d86410b9e2f34e7f065b38ecf6179d83d23a187b0a11ce48b13b53dc", "id": "B1.R1c", "kind": "str", "mistakes": {"698853838dd588357509a28a9621872955ee8966685339001ff534bee826c585": "un découpage non stratifié change les propor

**B1.R2 — Ch. 23 : geler un réseau et remplacer sa tête en PyTorch**

In [15]:
wb.record("B1.R2a", "B", mistakes={"eval() change le comportement de la batchnorm et du dropout, pas requires_grad": "A", "no_grad autour de step() ne gèle rien : les gradients des poids sont déjà calculés": "C"})
wb.record("B1.R2b", False, mistakes={"eval() change le mode de calcul de certaines couches : les poids reçoivent encore leur gradient": True})
wb.record("B1.R2c", 1280 * 4 + 4, mistakes={"n'oublie pas les 4 biais": 1280 * 4,
          "la nouvelle tête a 4 sorties, pas 1 000": 1280 * 1000 + 1000,
          "chaque sortie a un poids par entrée": 1280 + 4})
wb.record("B1.R2d", True, mistakes={"une couche neuve ne sait rien du gel des autres : quelle est la valeur par défaut de requires_grad ?": False})
wb.record("B1.R2e", "A", mistakes={"weights_only=False laisse pickle exécuter du code caché dans le fichier": "B", "pickle.load exécute ce que le fichier lui demande : jamais sur un fichier reçu": "C"})

WB_ANSWER {"hash": "76a093ee99d524cc0bd7b658d85689368527542191d6ac44ff3e321e27a8b5e9", "id": "B1.R2a", "kind": "str", "mistakes": {"45ad2e355c7142e43e161bd20c1a822b35d617de493e7a2c87d79ac1646e332f": "no_grad autour de step() ne gèle rien : les gradients des poids sont déjà calculés", "b7695aec9d9e86aa8934265dc9e91b7e8290d2f4d5f6f8adf445cdbbfed2ae3d": "eval() change le comportement de la batchnorm et du dropout, pas requires_grad"}}
📝 Ex B1.R2a : réponse enregistrée (str).
WB_ANSWER {"hash": "b2dde57b969b994d1317533c937b84669e1bd8f5416d1679926b2e82918b0148", "id": "B1.R2b", "kind": "bool", "mistakes": {"fa638aa84274b29bc278169b7f043d363ba0b7487b88017257aa2f891093c98a": "eval() change le mode de calcul de certaines couches : les poids reçoivent encore leur gradient"}}
📝 Ex B1.R2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "8e466e758206c12f47911cd86bb5c867e86aa29b6c388aa95a633c1bf78d892b", "id": "B1.R2c", "kind": "int", "magnitude": 3, "mistakes": {"12a0aa4f11e2d4b979cba9e9fbc9b7518

**B1.R3 — Ch. 21 : ce que détectent les filtres des premières et des dernières couches**

In [16]:
wb.record("B1.R3a", "B", mistakes={"des parties d'objets apparaissent dans les couches profondes, pas dans conv1": "A", "les filtres d'un réseau entraîné sont structurés : relis 📦 21.27": "C"})
wb.record("B1.R3b", 64 * 3 * 7 * 7, mistakes={"les convolutions de ResNet n'ont pas de biais": 64 * 3 * 7 * 7 + 64,
          "chaque filtre voit les 3 canaux de l'image": 64 * 7 * 7,
          "il y a 64 filtres": 3 * 7 * 7})
wb.record("B1.R3c", True, mistakes={"chaque stride 2 divise le côté de la carte : combien y en a-t-il avant layer4 ?": False})
wb.record("B1.R3d", False, mistakes={"un filtre de layer4 a des centaines de canaux d'entrée : peut-on l'afficher comme une image en couleur ?": True})
wb.record("B1.R3e", "A", mistakes={"le pooling moyen global fait la moyenne sur les positions, pas sur les canaux": "B", "le pooling moyen global ne garde qu'une valeur par canal": "C"})

WB_ANSWER {"hash": "90fe2f52d55adec0d7564706effd6d81a5b1fa8983dec5c05b80dfbae68b4915", "id": "B1.R3a", "kind": "str", "mistakes": {"4a338bbd15628bc91e6989ddef324ba7c4aae02765fe4dbfd4515e34a736192b": "les filtres d'un réseau entraîné sont structurés : relis 📦 21.27", "968a6ba891065c4b018e9c7d3d953e5859f015ea071ecef39fd2cd5194f5bd42": "des parties d'objets apparaissent dans les couches profondes, pas dans conv1"}}
📝 Ex B1.R3a : réponse enregistrée (str).
WB_ANSWER {"hash": "3eb7ab6e5bce4d27f821a1dd4eba41b4dbb92e3a5a1c9c6ee097071f4b818c40", "id": "B1.R3b", "kind": "int", "magnitude": 3, "mistakes": {"4b77b2132f7df6f5b440408490b907031146b6def1654d2700728bb6cbe70fcf": "les convolutions de ResNet n'ont pas de biais", "7a0f59aeac2e7c1cb534f78f6aa309b8c3b59f7384ad2bacefc973bc319db662": "il y a 64 filtres", "d15da7be0bdc85d2735fe3757a97ce6b767dc8e1e1954c91fa580faa8ba692d0": "chaque filtre voit les 3 canaux de l'image"}, "sign": 1}
📝 Ex B1.R3b : réponse enregistrée (int).
WB_ANSWER {"hash": "cea

**Ex B1.1 — Nouvelle tête : formes des tenseurs de l'image 32×32 redimensionnée jusqu'aux logits**

In [17]:
wb.record("B1.1a", shape(16, 64, S128_B11["stem"], S128_B11["stem"]), mistakes={"le maxpool du stem divise aussi le côté par 2": "(16, 64, 64, 64)",
          "n'oublie pas l'axe des exemples": "(64, 32, 32)",
          "conv1 a un stride 2, puis maxpool aussi : relis le stem": "(16, 64, 128, 128)"})
wb.record("B1.1b", shape(16, 128, S128_B11["layer2"], S128_B11["layer2"]), mistakes={"layer2 commence par une convolution de stride 2": "(16, 128, 32, 32)",
          "layer1 garde la taille : relis l'énoncé": "(16, 128, 8, 8)",
          "relis le nombre de canaux de layer2 dans l'énoncé": "(16, 256, 16, 16)"})
wb.record("B1.1c", shape(16, 512, S128_B11["layer4"], S128_B11["layer4"]), mistakes={"compte les strides 2 de layer2, layer3 et layer4": "(16, 512, 8, 8)",
          "layer1 ne divise pas le côté de la carte": "(16, 512, 2, 2)",
          "n'oublie pas l'axe des exemples": "(512, 4, 4)"})
wb.record("B1.1d", shape(16, 512), mistakes={"c'est la sortie d'avgpool : n'oublie pas l'aplatissement": "(16, 512, 1, 1)",
          "avgpool fait d'abord la moyenne de chaque canal sur toutes ses positions": "(16, 8192)",
          "n'oublie pas l'axe des exemples": "(512,)"})
wb.record("B1.1e", shape(16, 10), mistakes={"la tête d'ImageNet a été remplacée par une tête à 10 classes": "(16, 1000)",
          "n'oublie pas l'axe des exemples": "(10,)",
          "relis la question : on demande la sortie de fc": "(16, 512)"})
wb.record("B1.1f", resnet18_sides(32)["layer4"], mistakes={"compte toutes les étapes de stride 2, celles du stem comprises": 4,
          "une étape a été oubliée : conv1 et maxpool divisent chacun le côté": 2,
          "une convolution avec padding garde au moins une position : applique la formule à la dernière étape": 0},
          fractional="le côté d'une carte est un nombre entier : applique la formule à chaque étape")
wb.record("B1.1g", resnet18_sides(100)["layer4"], mistakes={"ni 100/32, ni des divisions par 2 arrondies vers le bas : applique la formule de l'énoncé à chaque étape, avec son padding": 3,
          "compte toutes les étapes de stride 2, celles du stem comprises": 13,
          "une étape de stride 2 a été oubliée": 7},
          fractional="le côté d'une carte est un nombre entier : applique la formule à chaque étape")
wb.record("B1.1h", True, mistakes={"quelle que soit la taille de l'image, que rend le pooling moyen adaptatif ?": False})

WB_ANSWER {"hash": "8183aa718c1f2a393e9cd011a9eebed493e6b21d4d313d7db0ab5669ac777487", "id": "B1.1a", "kind": "str", "mistakes": {"0b286250cc07fadfd0fd1d0ef3e98a46b7c951ecca62cfa073917edc462bf687": "le maxpool du stem divise aussi le côté par 2", "2b9f1c7add9dd27d432cf54af0ad3b4fad6280d1b8a3108b3af1866dd7a6276c": "conv1 a un stride 2, puis maxpool aussi : relis le stem", "f463343fad38082ef7ff80de3a7f271e37f0c233a3d7be84c76f9c1b74e0e586": "n'oublie pas l'axe des exemples"}}
📝 Ex B1.1a : réponse enregistrée (str).
WB_ANSWER {"hash": "8bcfea0bb7a278c817bfa0ce33464944f0198dbff9ec933f281e140a504383b7", "id": "B1.1b", "kind": "str", "mistakes": {"329f3ce68be06ad764a0834d214b732cce2c29af3e5ea3358e5dfb72d2c48571": "layer2 commence par une convolution de stride 2", "3593fb0bf3e5dba94e067a5d59ad5a320360a1057c73217beec21150181ecf34": "layer1 garde la taille : relis l'énoncé", "d8c1673fbdee34b2110c04ab9291bfdd15098bdb855290ad22a3f0b2eada326a": "relis le nombre de canaux de layer2 dans l'énoncé"}}


**Ex B1.2 — k plus proches voisins cosinus sur six embeddings donnés**

In [18]:
wb.record("B1.2a", COS_B12[5], decimals=3, mistakes={"un cosinus ne dépasse pas 1 : divise aussi par la norme de e5": 4 / 3,
          "un cosinus ne dépasse pas 1 : divise aussi par la norme de q": 4 / np.sqrt(2),
          "le produit scalaire seul n'est pas le cosinus : divise par les normes": 4.0})
wb.record("B1.2b", int(np.argmax(COS_B12)), mistakes={"e0 n'a pas la même direction que q : recalcule son cosinus": 0, "e1 est tout près : calcule les deux meilleurs cosinus à 3 décimales": 1, "relis la question : le cosinus divise le produit scalaire par les normes": 2, "c'est le plus proche en distance euclidienne : on demande le cosinus": 3, "e4 a un petit cosinus avec q : recalcule-le": 4, "il n'y a que six vecteurs, numérotés de 0 à 5": 6}, fractional="on demande le numéro d'un vecteur, de 0 à 5")
wb.record("B1.2c", "avion", mistakes={"à égalité de votes, la règle de l'énoncé ne regarde pas la distance des voisins": "oiseau"})
wb.record("B1.2d", "oiseau", mistakes={"avec k = 3, compte les votes des trois plus proches : y a-t-il encore égalité ?": "avion"})
wb.record("B1.2e", int(np.argmax(E_B12 @ Q_B12)), mistakes={"recalcule le produit scalaire de e0 avec q, et compare-le à tous les autres": 0, "recalcule les six produits scalaires : lequel est le plus grand ?": 1, "e3 a un produit scalaire moyen : compare les six": 3, "e4 a le plus petit produit scalaire, pas le plus grand": 4, "relis la question : on ne divise pas par les normes": 5, "il n'y a que six vecteurs, numérotés de 0 à 5": 6}, fractional="on demande le numéro d'un vecteur, de 0 à 5")
wb.record("B1.2f", float(((Q_B12 / 3 - E_B12[1] / 5) ** 2).sum()), decimals=3, mistakes={"relis la relation de la fiche entre distance au carré et cosinus pour des vecteurs de norme 1 : un facteur manque": 1 - 14 / 15,
          "on demande la distance au carré, pas la distance": np.sqrt(2 - 2 * 14 / 15),
          "ramène les deux vecteurs à la norme 1 avant de soustraire": 6.0})
wb.record("B1.2g", False, mistakes={"après cette division, compare encore les produits scalaires des autres vecteurs avec q : sont-ils normalisés ?": True})

WB_ANSWER {"decimals": 3, "hash": "171bee608cab46589cc365f392498ccdf25a31db8245dd14c292280d54b6b1a7", "hash_coarse": "d0ba4cb1b921d7986e85c8086e1c6fc1dbc49c027ee6e9f811b663558f577f50", "id": "B1.2a", "kind": "float", "magnitude": -1, "mistakes": {"5ffe399bf75c361d010658e820868a6e68ad11494f912a98fc78a9586c57af36": "un cosinus ne dépasse pas 1 : divise aussi par la norme de q", "f165bc6754a61bad0646cba159ddb817f65bbb75bb593f433a57f10266529964": "le produit scalaire seul n'est pas le cosinus : divise par les normes", "f4d7a48ed19b286bf2354b777111005e396e323d90a6ae6c6ca57dac55840375": "un cosinus ne dépasse pas 1 : divise aussi par la norme de e5"}, "sign": 1}
📝 Ex B1.2a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"fractional": "on demande le numéro d'un vecteur, de 0 à 5", "hash": "2acdc3a4876aed82d5d0dbe34fd21283331012879f97cea517b85f5af8a06f42", "id": "B1.2b", "kind": "int", "magnitude": 0, "mistakes": {"2a6e3a325071390ddb9d08d79ad62cbc8dc9801b9d15e9b21e41b74d8ec2d972": "re

**Ex B1.3 — Learning rates discriminatifs et warmup : le learning rate de chaque groupe, pas à pas**

In [19]:
wb.record("B1.3a", lr_b13(3, T_WARMUP_B13), decimals=1, mistakes={"compte les puissances de γ entre la tête et layer2 : il y en a une de trop": 100 * 0.3 ** 4,
          "compte les puissances de γ entre la tête et layer2 : il en manque une": 100 * 0.3 ** 2,
          "donne la valeur en unités de 10⁻⁵ : 10⁻³ s'écrit 100": 2.7e-5})
wb.record("B1.3b", round(lr_b13(0, 2)), mistakes={"les pas se comptent à partir de 0 : au pas 0, le facteur du warmup vaut 0": 60,
          "au pas 2, le warmup n'est pas fini": 100})
wb.record("B1.3c", round(lr_b13(1, 3)), mistakes={"les pas se comptent à partir de 0 : au pas 0, le facteur du warmup vaut 0": 24,
          "au pas 3, le warmup n'est pas fini": 30,
          "relis la question : on demande layer4": 60})
wb.record("B1.3d", lr_b13(0, 3) / lr_b13(3, 3), decimals=2, mistakes={"compte les puissances de γ entre la tête et layer2 : il y en a une de trop": 0.3 ** -4,
          "compte les puissances de γ entre la tête et layer2 : il en manque une": 0.3 ** -2,
          "le rapport est inversé : on divise le learning rate de la tête par celui de layer2": 0.3 ** 3})
wb.record("B1.3e", lr_b13(2, 55), decimals=2, mistakes={"au pas 55, la décroissance en cosinus a commencé": 9.0,
          "la progression du cosinus se compte à partir de la fin du warmup": 9 * 0.5 * (1 + math.cos(math.pi * 55 / 105)),
          "relis la question : on demande layer3": 50.0})
wb.record("B1.3f", lr_b13(0, 30), decimals=2, mistakes={"la progression du cosinus se compte à partir de la fin du warmup": 100 * 0.5 * (1 + math.cos(math.pi * 30 / 105)),
          "relis le signe devant le cosinus dans f(t)": 100 * 0.5 * (1 - math.cos(math.pi / 4)),
          "n'oublie pas le facteur ½ et le 1 de f(t)": 100 * math.cos(math.pi / 4)})
wb.record("B1.3g", True, mistakes={"relis f(0), puis comment AdamW applique son weight decay découplé (ch. 19)": False})
wb.record("B1.3h", False, mistakes={"relis la fiche : que fait LambdaLR de la valeur que rend la fonction ?": True})

WB_ANSWER {"decimals": 1, "hash": "dc812610e50fe4fa8a724d958889f887ba5672aec5215a215bf7b4f7a2af5bbd", "hash_coarse": "4d42bdf3ea7f05f37db7ff74ba6e08055b89c375ba66c67b31ff2a46153b0e17", "id": "B1.3a", "kind": "float", "magnitude": 0, "mistakes": {"253a75f11f0b64ba2ed1e2aea29ca13ce541e4473b4fb4bf66b57798d314c279": "donne la valeur en unités de 10⁻⁵ : 10⁻³ s'écrit 100", "ae5a075f969cec3e613c898ce179a88a79369e7ed3a3b12c108e43d3facc3272": "compte les puissances de γ entre la tête et layer2 : il y en a une de trop", "bccaf1f47ba539072976afbf60da136fec854c9c8cde4a92d67be35fec2231fb": "compte les puissances de γ entre la tête et layer2 : il en manque une"}, "sign": 1}
📝 Ex B1.3a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"hash": "6ef14f1f76e62143948c3cdd58d7df5d9241c553dd1f757120b0f91ed8b9747c", "id": "B1.3b", "kind": "int", "magnitude": 1, "mistakes": {"16484871fb05af8d73448af8d4daf41db1efc048ac628fe54b1fab683bde605b": "les pas se comptent à partir de 0 : au pas 0, le facteur du

**Ex B1.4 — Compter les paramètres entraînables : tête seule, dernier bloc, réseau entier**

In [20]:
wb.record("B1.4a", HEAD10_B14, mistakes={"n'oublie pas les 10 biais": 5120,
          "la tête d'ImageNet a été remplacée : la nouvelle a 10 sorties": 513000,
          "chaque sortie a un poids par composante de l'embedding": 522})
wb.record("B1.4b", NET10_B14, mistakes={"retire l'ancienne tête avant d'ajouter la nouvelle": IMAGENET_B14 + HEAD10_B14,
          "ajoute la nouvelle tête au backbone": IMAGENET_B14 - 513000,
          "c'est le réseau avec sa tête ImageNet : remplace-la": IMAGENET_B14})
wb.record("B1.4c", conv_weights(512, 512, 3), mistakes={"il y a 512 filtres, un par canal de sortie": 3 * 3 * 512,
          "n'oublie pas la taille du noyau, 3 × 3": 512 * 512,
          "les convolutions de ResNet n'ont pas de biais": conv_weights(512, 512, 3) + 512})
wb.record("B1.4d", LAYER4_B14, mistakes={"n'oublie pas le raccourci (downsample) du bloc 0": LAYER4_B14 - conv_weights(256, 512, 1) - bn_params(512),
          "n'oublie pas les batchnorms : deux paramètres par canal": LAYER4_B14 - 5 * bn_params(512),
          "les statistiques glissantes sont des buffers, pas des paramètres": LAYER4_B14 + 5 * bn_params(512),
          "la première convolution du bloc 0 reçoit 256 canaux, pas 512": LAYER4_B14 - conv_weights(256, 512, 3) + conv_weights(512, 512, 3),
          "layer4 a deux blocs": BLOCK1_B14})
wb.record("B1.4e", TRAIN_B14, mistakes={"ajoute la tête, entraînée elle aussi": LAYER4_B14,
          "la tête d'ImageNet a été remplacée par une tête à 10 classes": LAYER4_B14 + 513000})
wb.record("B1.4f", 100 * TRAIN_B14 / NET10_B14, decimals=1, mistakes={"divise par le nombre de paramètres du réseau muni de la nouvelle tête": 100 * TRAIN_B14 / IMAGENET_B14,
          "divise par le nombre de paramètres du réseau muni de sa nouvelle tête, pas de l'ancienne": 71.9,
          "on demande un pourcentage, pas une proportion": TRAIN_B14 / NET10_B14,
          "c'est la part des paramètres gelés": 100 - 100 * TRAIN_B14 / NET10_B14})
wb.record("B1.4g", TRAIN_B14 - 5 * bn_params(512), mistakes={"gèle aussi les paramètres des batchnorms de layer4": TRAIN_B14,
          "chaque batchnorm a deux paramètres par canal : ne les retire qu'une fois": TRAIN_B14 - 10 * bn_params(512),
          "n'oublie pas la batchnorm du raccourci": TRAIN_B14 - 4 * bn_params(512),
          "la nouvelle tête, elle, reste entraînable": TRAIN_B14 - 5 * bn_params(512) - HEAD10_B14,
          "relis la question : on part de e), où la tête est entraînée elle aussi": LAYER4_B14})
wb.record("B1.4h", False, mistakes={"eval() change le calcul des batchnorms : change-t-il leur requires_grad ?": True})

WB_ANSWER {"hash": "43d8c235757441640a6800d0b9841227a3291f479bd06a4af0013fc670f9a95d", "id": "B1.4a", "kind": "int", "magnitude": 3, "mistakes": {"009dd110bf11487d166d9055a99bb4c80bec52f41b84cafd4c7e9fe5ae852b8a": "la tête d'ImageNet a été remplacée : la nouvelle a 10 sorties", "60dc2447a7f70127b31f681c850050a5e7615898b7b788d03b08dba773aaf3f7": "n'oublie pas les 10 biais", "fdc3164f5a693ae923e2707de2dab9200cf05ac29ceef05f605f980541bb99f2": "chaque sortie a un poids par composante de l'embedding"}, "sign": 1}
📝 Ex B1.4a : réponse enregistrée (int).
WB_ANSWER {"hash": "d7522cdcbb8f1fbdb94c050a0307e1ca39a840c1aaab8ca3434947c7c74d54e5", "id": "B1.4b", "kind": "int", "magnitude": 7, "mistakes": {"91e9eded1a7562ad6d5c60b221222057ce2aa5e0cfd107a021a5c30a3517c351": "c'est le réseau avec sa tête ImageNet : remplace-la", "bfcc3384adbac8df864643f6d1c3eaf011802d2d3be661f05ff4bf0076dc25b9": "retire l'ancienne tête avant d'ajouter la nouvelle", "e1609e842541a6d5e0a69ca51d7abbabb64fb4d8734657a02365ee

**Ex B1.5 — Zéro-shot CLIP à la main : cosinus, logit_scale et softmax**

In [21]:
wb.record("B1.5a", P100_B15[1], decimals=3, mistakes={"multiplie les cosinus par s avant la softmax": zero_shot_b15(1.0)[1],
          "s est l'exponentielle de logit_scale": zero_shot_b15(4.60517)[1],
          "la softmax prend l'exponentielle de chaque logit": COS_B15[1] / COS_B15.sum(),
          "relis la question : s vient ici du paramètre entraîné": PINIT_B15[1]})
wb.record("B1.5b", P100_B15[2], decimals=3, mistakes={"multiplie les cosinus par s avant la softmax": zero_shot_b15(1.0)[2],
          "s est l'exponentielle de logit_scale": zero_shot_b15(4.60517)[2],
          "la softmax prend l'exponentielle de chaque logit": COS_B15[2] / COS_B15.sum(),
          "relis la question : s vient ici du paramètre entraîné": PINIT_B15[2]})
wb.record("B1.5c", P100_B15[1] / P100_B15[2], decimals=3, mistakes={"le rapport des probabilités n'est pas celui des cosinus": 0.30 / 0.28,
          "n'oublie pas l'exponentielle": 2.0,
          "le rapport est inversé : p(oiseau) au numérateur": P100_B15[2] / P100_B15[1],
          "multiplie l'écart des cosinus par s": math.exp(0.02),
          "tu as divisé des probabilités déjà arrondies : calcule le rapport sans arrondir en chemin": round(P100_B15[1], 3) / round(P100_B15[2], 3)})
wb.record("B1.5d", P100_B15[0], decimals=4, mistakes={"divise par la somme des exponentielles": math.exp(-6),
          "multiplie les cosinus par s avant la softmax": zero_shot_b15(1.0)[0],
          "relis la question : s vient ici du paramètre entraîné": PINIT_B15[0]})
wb.record("B1.5e", PINIT_B15[1], decimals=2, mistakes={"relis la question : s vient ici de la valeur initiale": P100_B15[1],
          "s est l'exponentielle de la valeur initiale, pas 1": zero_shot_b15(1.0)[1],
          "s est l'exponentielle de logit_scale_init_value": zero_shot_b15(2.6592)[1]})
wb.record("B1.5f", True, mistakes={"écris le rapport des deux probabilités : que devient le dénominateur commun ?": False})

WB_ANSWER {"decimals": 3, "hash": "5cb45c6ee022a0e8dd5650cf0e6ed18e652987ef3f267dc4045fa224b003e841", "hash_coarse": "d75b615e58cb2ea55a2ab9f65450d7b0bff1b3ff976b50ed54082a8d48c56580", "id": "B1.5a", "kind": "float", "magnitude": -1, "mistakes": {"5d39916f134f0dbdb45aea36742924f0a9ad4a5099edbe64c99976824318aca7": "relis la question : s vient ici du paramètre entraîné", "957e23aa34dcf2ae875c9644539dd300f744ffef199c8981d9e7a279c3db0253": "multiplie les cosinus par s avant la softmax", "c2838c1cdcd4068de0b3ee02b9961f3bfc9f53ce79796d6651d751073b3f660d": "s est l'exponentielle de logit_scale", "ef9ccf30925b059b6a051b5b8b7a6750d35b3903ef1bea4f709a2102f39e19f0": "la softmax prend l'exponentielle de chaque logit"}, "sign": 1}
📝 Ex B1.5a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "7783f13b3c6f8e1cb74b3d97210d845d5a18d282235eb90e38aa8801f6be8ecf", "hash_coarse": "3a21a89c782690689338a61ff55ddd6b28e28a754b8ebb7a46f9a2fbd20ce8f9", "id": "B1.5b", "kind": "float"

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu suivre les formes d'un mini-batch dans un ResNet pré-entraîné et compter ce que tu entraînes selon la stratégie ?
2. Sais-tu écrire les groupes de paramètres et le planning d'un fine-tuning, et dire pourquoi les premières couches bougent le moins ?
3. Sais-tu calculer les probabilités d'un zéro-shot de CLIP, et dire ce que fait l'échelle $s$ ?
4. Sais-tu dire ce qu'un modèle dérivé hérite de son modèle de base (licence, biais, limites) ?

**Pour aller plus loin** : le tutoriel de PyTorch sur le transfer learning et l'article de Yosinski et coll. (2014), cités dans la fiche. La suite : le bonus B2, tokenisation et embeddings.

*Les exercices de code B1.11 à B1.28 (parties A à D : charger et sonder, fine-tuner, quand le transfert aide ou échoue, modèles de fondation, coût et livraison) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B1` pour obtenir le notebook complet.*